In [ ]:
import torch
from transformers import AutoTokenizer, TrainingArguments, AutoModelForSequenceClassification, Trainer, BertConfig
from datasets import Dataset
import random
import pandas as pd
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.tensorboard import SummaryWriter
from transformers.integrations.integration_utils import TensorBoardCallback

# Set the random seed for reproducibility
def set_seed(seed=42): # As per Hitchhiker's Guide to the Galaxy, 42 is the answer to life, the universe, and everything
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed() 

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = pd.read_csv('./data/imdb_data.csv')
training_set = Dataset.from_pandas(data[data['dataset'] == 'train'])
test_set = Dataset.from_pandas(data[data['dataset'] == 'test'])

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # this tokenizer will work for our smaller model

# Helper function to tokenize the text data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="longest", truncation=True)

# Tokenize the training and test sets
tokenized_training_set = training_set.map(tokenize_function, batched=True)
tokenized_test_set = test_set.map(tokenize_function, batched=True)

# Training arguments for the Trainer
training_args = TrainingArguments(
    output_dir="./temp_results", # do not change this
    num_train_epochs=3,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="none", # Do not report to any logging or tracking system. Logging to be done later using TensorBoard. 
    #Fix needed due to missing logging_dir parameter in TrainingArguments for newer versions of the Transformers library.
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="no", # do not change this
)

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

In [ ]:
# LoRA configuration for fine-tuning the model
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence classification task
    inference_mode=False,  # False during training, True during inference
    r=16,  # Rank of the low-rank matrices
    lora_alpha=32,  # Analogous to the learning rate in standard gradient descent, typically set to double the rank
    lora_dropout=0.1  # Dropout rate to help prevent overfitting
)

In [4]:
# Ensure that the model configuration is correctly loaded, considering missing attributes in the config.json of the pretrained model.

model_id = "prajjwal1/bert-tiny" # Identifier for the pretrained BERT model to be used.
config = BertConfig.from_pretrained(model_id) # Load the BERT model configuration with necessary adjustments.
config.num_labels = 2 # Set the number of labels for sequence classification.

model = AutoModelForSequenceClassification.from_pretrained(model_id, config=config) # Load the pretrained model with the adjusted configuration.
lora_model = get_peft_model(model, peft_config) # Apply the LoRA configuration to the model.

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

In [5]:
# Function to print the number of trainable parameters in the model
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )
    
print_trainable_parameters(lora_model)

trainable params: 16642 || all params: 4402820 || trainable%: 0.37798501869256523


In [6]:
# put the model on the GPU
lora_model.to(device)

# initialize the Trainer with the LoRA model and training arguments
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_training_set,
    eval_dataset=tokenized_test_set,
    callbacks=[TensorBoardCallback(tb_writer=SummaryWriter(log_dir="./logs"))],
)

# start training and evaluation
trainer.train()
trainer.evaluate()

Step,Training Loss
10,0.687592
20,0.691262
30,0.692680
40,0.690232
50,0.697249
60,0.693240
70,0.695972
80,0.694207
90,0.697538
100,0.698519


Training Loss,Validation Loss,Step
0.696534,0.680063,315


{'eval_loss': 0.6800633668899536}

In [7]:
lora_model.save_pretrained("./lora_finetunedmodel")